In [1]:
# Access your secret keys via
from google.colab import userdata
# The name of your secret must match `secretName`
OPENAI_API_KEY = userdata.get('chat_GPT')

# Import OpenAI API and set up the key
from openai import OpenAI
client = OpenAI(api_key=(OPENAI_API_KEY))

In [7]:
# In Google Colab, install the OpenAI package first:
# %pip install -q openai

from google.colab import userdata
from openai import OpenAI
import re

client = OpenAI(api_key=userdata.get("chat_GPT"))
MODEL = "gpt-4.1-mini"  # Or "gpt-4.1-nano"


def square_with_python(number):
    """Calculate the exact square using Python."""
    return number * number


def square_with_chatgpt(number):
    """Ask the model to calculate the square and parse its integer answer."""
    response = client.responses.create(
        model=MODEL,
        input=(
            "Calculate the exact integer square of this number. "
            "Return only the answer as a base-10 integer.\n"
            f"Number: {number}"
        ),
        temperature=0,
    )

    answer = response.output_text.strip().replace(",", "")
    if not re.fullmatch(r"-?\d+", answer):
        raise ValueError(f"Model did not return a plain integer: {answer!r}")

    return int(answer)


def run_experiment(n, iterations):
    current = n
    error_count = 0

    for step in range(1, iterations + 1):
        # Both methods square the same current value.
        python_result = square_with_python(current)

        try:
            chatgpt_result = square_with_chatgpt(current)
            matches = chatgpt_result == python_result
        except Exception as error:
            chatgpt_result = None
            matches = False
            print(f"ChatGPT/API error: {error}")

        print(f"\nStep {step}: square {current}")
        print(f"Python result:   {python_result}")
        print(f"ChatGPT result:  {chatgpt_result}")

        if matches:
            print("ChatGPT got it right.")
        else:
            messages = [
                "Wow, I messed up again. Math is hard, okay?",
                "Another wrong square. I think the numbers are winning.",
                "That's another miss. Please stop asking me to square things.",
            ]
            print(messages[min(error_count, len(messages) - 1)])
            error_count += 1

        # Continue from Python's correct answer so the reference stays exact.
        current = python_result

    print(f"\nFinal result: {current}")
    return current


# Get values from the user.
try:
    n = int(input("Enter the base integer n: "))
    iterations = int(input("Enter the number of iterations i: "))

    if iterations < 0:
        raise ValueError("The number of iterations must be zero or greater.")

    run_experiment(n, iterations)

except ValueError as error:
    print(f"Invalid input: {error}")

Enter the base integer n: 5
Enter the number of iterations i: 6

Step 1: square 5
Python result:   25
ChatGPT result:  25
ChatGPT got it right.

Step 2: square 25
Python result:   625
ChatGPT result:  625
ChatGPT got it right.

Step 3: square 625
Python result:   390625
ChatGPT result:  390625
ChatGPT got it right.

Step 4: square 390625
Python result:   152587890625
ChatGPT result:  152587890625
ChatGPT got it right.

Step 5: square 152587890625
Python result:   23283064365386962890625
ChatGPT result:  23283064365386962890625
ChatGPT got it right.

Step 6: square 23283064365386962890625
Python result:   542101086242752217003726400434970855712890625
ChatGPT result:  542202927237360111096954692535400390625
Wow, I messed up again. Math is hard, okay?

Final result: 542101086242752217003726400434970855712890625
